# Beyond MASE: Spike Detection and Interval Calibration in Zero-Shot $\text{PM}_{2.5}$ Forecasting for Indian Cities

**Chaitanya Jhade, Atharva Khewle, Prasad Satpute, Hringkesh Singh, Himani Deshpande**  
*Department of Artificial Intelligence & Data Science, Thadomal Shahani Engineering College, Mumbai, India*  
Contact: chaitanyajhade99@gmail.com, Himani.deshpande@thadomal.org

---

### Abstract
Time-series foundation models such as Chronos and Chronos Bolt can forecast a new series without being trained on it, which makes them attractive for cities that lack the resources to train and maintain their own air quality models. This paper evaluates three zero-shot Chronos family models against seven trained and naive baselines for daily $\text{PM}_{2.5}$ forecasting across nine Indian cities, using rolling-origin, time-ordered splits over three test years including the 2020 lockdown period. Beyond the mean absolute scaled error that most forecasting papers report, we measure two properties that matter for a public health deployment: how often each model detects a day that exceeds the city\'s own high pollution threshold, and how often its uncertainty interval actually contains the true value, broken down by season, lockdown status, and pollution level. 

The best zero-shot model, Chronos 2, achieves the lowest average error at every horizon and about nineteen percent lower error than persistence at seven days ahead, yet at one day ahead its ability to flag high pollution days is well behind simple persistence, and every model\'s default uncertainty interval covers only about half of true high pollution days rather than the intended eighty percent. A level-scaled conformal calibration step restores interval coverage on these days to around seventy percent or higher across models and horizons, with one case (gradient boosting at seven days) falling just under at sixty-nine percent. The results indicate that average error and public health usefulness are not the same thing, and that reporting either alone is not enough for city-scale deployment decisions.

---

### Index Terms
Air quality forecasting, conformal prediction, foundation models, $\text{PM}_{2.5}$, time series forecasting, zero-shot learning.

## Section I: Introduction & Software Environment

Atmospheric particulate matter of diameter $\le 2.5\,\mu\text{m}$ ($\text{PM}_{2.5}$) represents one of the most hazardous environmental risk factors globally, with particularly severe manifestations across urban centers in the Indo-Gangetic Plain and South Asia. Traditional numerical weather prediction (NWP) coupled with chemical transport models (CTMs) suffers from prohibitive computational complexity and high sensitivity to uncertain emission inventories. While statistical and deep learning approaches have advanced operational forecasting, they typically require extensive local historical training data. Recently, **Time-Series Foundation Models (TSFMs)** pretrained on diverse cross-domain datasets offer the promise of zero-shot forecasting without local parameter tuning.

This notebook executes an end-to-end reproducible pipeline comparing zero-shot foundation models against domain-trained machine learning architectures on Indian ambient air quality records.

```
Hardware Requirement: NVIDIA T4 GPU or higher (Google Colab Runtime -> Change runtime type -> T4 GPU)
Primary Packages: chronos-forecasting, torch, scipy, scikit-learn, pandas, numpy
```

In [ ]:
%pip install -q chronos-forecasting scipy

## Section II: Experimental Protocol & Deterministic Setup

To ensure strict experimental reproducibility across hardware environments, deterministic seeds are established across the pseudo-random number generators of Python standard library, NumPy, and PyTorch. 

### Experimental Parameters ($CFG$):
* **Forecast Horizons:** $h \in \{1, 3, 7\}$ days ahead.
* **Temporal Cross-Validation Folds:**
  * **Fold 2018:** Train: $[0, \text{2017-12-31}]$, Calibration: $[\text{2017-01-01}, \text{2017-12-31}]$, Test: $[\text{2018-01-01}, \text{2018-12-31}]$.
  * **Fold 2019:** Train: $[0, \text{2018-12-31}]$, Calibration: $[\text{2018-01-01}, \text{2018-12-31}]$, Test: $[\text{2019-01-01}, \text{2019-12-31}]$.
  * **Fold 2020:** Train: $[0, \text{2019-12-31}]$, Calibration: $[\text{2019-01-01}, \text{2019-12-31}]$, Test: $[\text{2020-01-01}, \text{2020-06-30}]$ (capturing the national COVID-19 lockdown shock).
* **Conformal Coverage:** $1 - \alpha \in \{0.80, 0.90\}$ corresponding to 80% and 90% confidence bands.
* **Cluster Bootstrap:** $B = 1000$ iterations clustered at the city level.

In [ ]:
# %% [1] Setup
import os, sys, io, json, time, zipfile, warnings, math, random, platform, pickle
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

CFG = dict(
    DATA_PATH=None,                       # path to the zip/csv; None -> upload dialog in Colab
    OUT="results",
    SMOKE=False,                          # tiny fast run for debugging
    MOCK_TSFM=False,                      # use a fake pipeline (plumbing test only, NEVER for results)
    CITY_START_BEFORE="2017-07-01", CITY_MIN_COV=0.89,      # fixed BEFORE looking at any result
    HORIZONS=[1, 3, 7],
    FOLDS=[("2018-01-01", "2018-12-31"), ("2019-01-01", "2019-12-31"), ("2020-01-01", "2020-06-30")],
    CAL_DAYS=365, MIN_TRAIN=365, MIN_CAL=200, MIN_TEST=20,
    LOCKDOWN=("2020-03-25", "2020-05-31"),
    ALPHAS=[0.2, 0.1],                    # 0.2 -> 80% intervals (matches TSFM q10-q90); 0.1 -> 90%
    ROLL_WIN=365, MIN_STRATUM=30, SCALE_FLOOR=10.0,   # ug/m3 floor for the level-normalised score
    TSFMS=["chronos-bolt-small", "chronos-bolt-base", "chronos-2"],
    RUN_TIMESFM=False,                    # optional, UNTESTED, weights are non-commercial licence
    CTX=512, MIN_CTX_OBS=14, TSFM_BATCH=64,
    LSTM=dict(enabled=True, seeds=[0, 1, 2], epochs=40, patience=6, hidden=64, L=30, lr=2e-3, bs=256, val_days=90),
    BOOT=2000, SEED=0,
)
if os.environ.get("PM25_SMOKE"): CFG.update(SMOKE=True, MOCK_TSFM=True, DATA_PATH=os.environ["PM25_DATA"], TSFMS=["chronos-bolt-small"], OUT="smoke_results")
if os.environ.get("PM25_OVERRIDE"): CFG.update(json.loads(os.environ["PM25_OVERRIDE"]))     # testing hook only
if CFG["SMOKE"]:
    CFG["LSTM"].update(seeds=[0], epochs=3, patience=2); CFG["BOOT"] = 300
# To survive Colab disconnects, save to Drive instead:
# from google.colab import drive; drive.mount("/content/drive"); CFG["OUT"] = "/content/drive/MyDrive/pm25_results"
os.makedirs(CFG["OUT"], exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception as e:
    torch, DEVICE = None, "cpu"; print("torch not available:", e)
log("device:", DEVICE, "| torch:", getattr(torch, "__version__", None))


## Section III: Ambient Air Quality Dataset & Ingestion

The empirical study utilizes official ground-level ambient monitoring data from the **Central Pollution Control Board (CPCB)** of India, archived as `city_day.csv`. The dataset records daily average concentrations of criteria pollutants:
$$\mathcal{P} = \{\text{PM}_{2.5}, \text{PM}_{10}, \text{NO}_2, \text{CO}, \text{SO}_2, \text{O}_3\}$$

The ingestion module supports direct upload in Google Colab (uploading `archive.zip` or `city_day.csv`) or reading from a local filesystem path specified in `CFG["DATA_PATH"]`.

In [ ]:
# %% [2] Load data
def load_city_day(path):
    if path is None:
        if IN_COLAB:
            from google.colab import files
            up = files.upload(); path = next(iter(up))
        else:
            raise SystemExit("Set CFG['DATA_PATH'] to the zip or city_day.csv")
    if str(path).lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            name = [n for n in z.namelist() if n.lower().endswith("city_day.csv")][0]
            return pd.read_csv(z.open(name), parse_dates=["Date"])
    return pd.read_csv(path, parse_dates=["Date"])

raw = load_city_day(CFG["DATA_PATH"])
log("loaded", raw.shape, raw.Date.min().date(), "->", raw.Date.max().date(), "| cities:", raw.City.nunique())


## Section IV: Pre-Registered Data Audit & Quality Control Protocol

To eliminate publication bias and post-hoc data dredging, candidate cities are selected using **fixed, pre-registered inclusion criteria**:
1. **Sufficient Historical Span:** Continuous monitoring commenced on or before July 1, 2017 ($t_{\text{start}} \le \text{2017-07-01}$).
2. **High Temporal Completeness:** Non-missing $\text{PM}_{2.5}$ observation ratio $\ge 89\%$ over the entire monitoring period ($\text{cov} \ge 0.89$).

### Environmental Regime Stratification
Air quality dynamics in India exhibit sharp regime shifts. We stratify all evaluation points into three mutually exclusive regimes:
* **Winter Smog Regime (Nov 1 – Feb 28):** Severe meteorological temperature inversions and post-harvest biomass burning.
* **Lockdown Shock Regime (Mar 25 – May 31, 2020):** Abrupt anthropogenic activity suppression during the national COVID-19 pandemic response.
* **Baseline / Other Regime:** Standard non-monsoon and monsoon dispersion conditions.

In [ ]:
# %% [3] Audit + FIXED city-selection rule
g = raw.groupby("City").agg(first=("Date", "min"), last=("Date", "max"), pm25_obs=("PM2.5", "count"))
g["span_days"] = (g["last"] - g["first"]).dt.days + 1
g["pm25_cov"] = (g.pm25_obs / g.span_days).round(3)
CITIES = g[(g["first"] <= CFG["CITY_START_BEFORE"]) & (g.pm25_cov >= CFG["CITY_MIN_COV"])].index.tolist()
if CFG["SMOKE"]: CITIES = CITIES[:3]
print(g.sort_values("pm25_obs", ascending=False).to_string()); print("\nSELECTED CITIES:", CITIES)
g.to_csv(f"{CFG['OUT']}/city_audit.csv")

POLL = ["PM2.5", "PM10", "NO2", "CO", "SO2", "O3"]
PANEL = {c: raw[raw.City == c].set_index("Date")[POLL].sort_index().asfreq("D") for c in CITIES}
CIDX = {c: i for i, c in enumerate(CITIES)}; NC = len(CITIES)
HS = CFG["HORIZONS"]; HMAX = max(HS)
LOCK0, LOCK1 = pd.Timestamp(CFG["LOCKDOWN"][0]), pd.Timestamp(CFG["LOCKDOWN"][1])

def regime_of(idx):
    idx = pd.DatetimeIndex(idx)
    lock = (idx >= LOCK0) & (idx <= LOCK1); win = (idx.month >= 11) | (idx.month <= 2)
    return np.where(lock, "lockdown", np.where(win, "winter", "other"))


## Section V: Multi-Horizon Temporal Partitioning & Feature Engineering

### Temporal Leakage Prevention
At any forecast origin date $t$, the models are strictly restricted to information available up to and including date $t$. The forecast target is the ground-truth value at date $t + h$, where $h \in \{1, 3, 7\}$.

### Feature Space Construction:
Let $y_t$ denote the ground-truth $\text{PM}_{2.5}$ concentration at date $t$. The feature vector $\mathbf{x}_t$ includes:
* **Log-Transform:** $l_t = \log(1 + y_t)$ and forward-filled persistence base $l_0 = \log(1 + y_{\text{ffill}})$.
* **Sensor Staleness:** Age $a_t$, defined as the number of consecutive calendar days since the last valid observation:
  $$a_t = t - \max \{\tau \le t : y_\tau \text{ is observed}\}$$
* **Recent Missingness Ratio:** $\text{miss}_{30}(t) = 1 - \frac{1}{30}\sum_{i=0}^{29} \mathbb{I}(y_{t-i} \text{ is observed})$.
* **Autoregressive Lags:** Historical log-concentrations at lags $k \in \{1, 2, 3, 6, 13, 20, 27\}$.
* **Multi-Scale Rolling Statistics:** Moving averages over windows $w \in \{3, 7, 14, 30\}$ days, rolling volatility (7-day standard deviation), and trend offset ($l_0 - \text{rm}_{30}$).
* **Calendar Seasonality:** Day-of-year cyclic trigonometric encodings:
  $$\sin\left(\frac{2\pi d}{365.25}\right), \quad \cos\left(\frac{2\pi d}{365.25}\right)$$

### Common Row Universe
To prevent survivor bias and guarantee statistical equity, evaluation is conducted exclusively on the **intersection of target dates** where all baseline, supervised, and foundation models provide valid predictions.

In [ ]:
# %% [4] Folds, features, row universe
FOLDS = []
for (a, b) in CFG["FOLDS"]:
    ts = pd.Timestamp(a); FOLDS.append(dict(name=a[:4], test_start=ts, test_end=pd.Timestamp(b),
                                            cal_start=ts - pd.Timedelta(days=CFG["CAL_DAYS"])))
if CFG["SMOKE"]: FOLDS = FOLDS[-2:]

def origin_feats(df):
    """Everything computed here is known at the ORIGIN day (day t-h). It is shifted by h later."""
    y = df["PM2.5"]; ly = np.log1p(y); obs = y.notna()
    F = pd.DataFrame(index=df.index)
    F["l0"] = np.log1p(y.ffill())                          # persistence (log space)
    F["age"] = (~obs).groupby(obs.cumsum()).cumsum().astype(float)          # days since last observation
    F["miss30"] = 1 - obs.astype(float).rolling(30, min_periods=1).mean()   # missing share, last 30 d
    F["ly0"] = ly
    for k in (1, 2, 3, 6, 13, 20, 27): F[f"ly_l{k}"] = ly.shift(k)
    for w in (3, 7, 14, 30): F[f"rm{w}"] = ly.rolling(w, min_periods=1).mean()
    F["rs7"] = ly.rolling(7, min_periods=3).std()
    F["trend"] = F["l0"] - F["rm30"]
    F["ma7_raw"] = y.rolling(7, min_periods=1).mean()
    for p in ("PM10", "NO2", "CO", "SO2", "O3"): F[f"lp_{p}"] = np.log1p(df[p].clip(lower=0))
    return F

ORIG = {c: origin_feats(PANEL[c]) for c in CITIES}
ML_COLS = None
def feats_h(c, h):
    X = ORIG[c].shift(h).copy()                             # row t now holds info available at t-h
    doy = X.index.dayofyear.to_numpy(); X["sin"] = np.sin(2*np.pi*doy/365.25); X["cos"] = np.cos(2*np.pi*doy/365.25)
    X["dow"] = X.index.dayofweek.to_numpy().astype(float)
    for j in range(NC): X[f"city_{j}"] = float(CIDX[c] == j)
    X["y"] = PANEL[c]["PM2.5"]; X["city"] = c; X["date"] = X.index
    return X
BIG = {h: pd.concat([feats_h(c, h) for c in CITIES], ignore_index=True) for h in HS}
ML_COLS = [k for k in BIG[HS[0]].columns if k not in ("y", "city", "date", "ma7_raw")]

# valid (city, fold) pairs are decided ONCE, independent of horizon/model
VALID = {}
for c in CITIES:
    y = PANEL[c]["PM2.5"]
    for f in FOLDS:
        ntr = int(y[y.index < f["cal_start"]].notna().sum()); nca = int(y[(y.index >= f["cal_start"]) & (y.index < f["test_start"])].notna().sum())
        nte = int(y[(y.index >= f["test_start"]) & (y.index <= f["test_end"])].notna().sum())
        VALID[(c, f["name"])] = (ntr >= CFG["MIN_TRAIN"]) and (nca >= CFG["MIN_CAL"]) and (nte >= CFG["MIN_TEST"])
log("valid city-folds:", sum(VALID.values()), "of", len(VALID))

def universe(c, f, h):
    """Rows (target days) we evaluate: calibration + test days with an observed target and a persistence forecast."""
    X = BIG[h][BIG[h].city == c].set_index("date")
    m = (X.index >= f["cal_start"]) & (X.index <= f["test_end"]) & X["y"].notna() & X["l0"].notna()
    U = X.loc[m, ["y", "l0", "age", "miss30", "ma7_raw"]].copy()
    U["split"] = np.where(U.index < f["test_start"], "calib", "test")
    tr = X[(X.index < f["cal_start"]) & X["y"].notna() & X["l0"].notna()]
    U.attrs["scale"] = float((tr["y"] - np.expm1(tr["l0"])).abs().mean())      # MASE scale: TRAIN persistence error
    U.attrs["p90"] = float(tr["y"].quantile(0.90))                              # 'spike' threshold from TRAIN only
    U.attrs["train_months"] = tr["y"].groupby(tr.index.month).mean().to_dict()
    return U

UNI = {}
for c in CITIES:
    for f in FOLDS:
        if not VALID[(c, f["name"])]: continue
        for h in HS: UNI[(c, f["name"], h)] = universe(c, f, h)
FOLD_BY_NAME = {f["name"]: f for f in FOLDS}


## Section VI: Benchmark Architectures & Classical Machine Learning

We formulate a comprehensive suite of domain-trained and heuristic baselines:

### 1. Heuristic & Statistical Benchmarks:
* **Persistence Baseline (Random Walk):**
  $$\hat{y}_{t+h|t} = y_t$$
* **7-Day Simple Moving Average (SMA-7):**
  $$\hat{y}_{t+h|t} = \frac{1}{7} \sum_{i=0}^6 y_{t-i}$$
* **Monthly Climatology:**
  $$\hat{y}_{t+h|t} = \bar{y}_{\text{month}(t+h), \text{train}}$$
* **Simple Exponential Smoothing (SES / ETS(A,N,N)):**
  $$\hat{y}_{t+1|t} = \alpha y_t + (1 - \alpha) \hat{y}_{t|t-1}$$
  where smoothing factor $\alpha \in [0.05, 0.95]$ is optimized via grid search over training folds.

### 2. Supervised Machine Learning (Residual Formulation):
To enforce stationarity and preserve physical boundaries, supervised models predict the **log-ratio residual relative to persistence**:
$$\Delta \hat{l}_{t+h} = \log\left(\frac{1 + y_{t+h}}{1 + y_t}\right), \quad \hat{y}_{t+h} = \exp\left(l_0 + \Delta \hat{l}_{t+h}\right) - 1$$
* **Ridge Regression (L2-Regularized):** Linear shrinkage model fitted with standard feature scaling.
* **Histogram-based Gradient Boosting (HistGB):** Tree-based ensemble with native missing-value routing and non-linear split partitioning.

In [ ]:
# %% [5] Trained baselines + naive baselines  (each returns Series indexed by target date)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

PRED = {}          # PRED[model][(city, fold, h)] -> pd.Series (index=target date)
def put(model, c, fn, h, s): PRED.setdefault(model, {})[(c, fn, h)] = s

# 5a naive: persistence, 7-day moving average, monthly climatology
for (c, fn, h), U in UNI.items():
    put("persist", c, fn, h, np.expm1(U["l0"]))
    put("ma7", c, fn, h, U["ma7_raw"])
    put("clim", c, fn, h, pd.Series([U.attrs["train_months"].get(m, np.nan) for m in U.index.month], index=U.index))

# 5b SES = ETS(A,N,N) implemented by hand; alpha picked on TRAIN only per (city, fold, horizon)
def ses_level(y, alpha):
    L = np.full(len(y), np.nan); cur = np.nan
    for i, v in enumerate(y):
        if not np.isnan(v): cur = v if np.isnan(cur) else alpha*v + (1-alpha)*cur
        L[i] = cur
    return L
for c in CITIES:
    yv = PANEL[c]["PM2.5"].to_numpy(); idx = PANEL[c].index
    for f in FOLDS:
        if not VALID[(c, f["name"])]: continue
        trm = (idx < f["cal_start"])
        for h in HS:
            best = (1e18, 0.5)
            for a in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
                L = ses_level(yv, a); pr = pd.Series(L, index=idx).shift(h).to_numpy()
                m = trm & ~np.isnan(yv) & ~np.isnan(pr); e = np.abs(yv[m] - pr[m]).mean() if m.sum() else 1e18
                if e < best[0]: best = (e, a)
            L = pd.Series(ses_level(yv, best[1]), index=idx).shift(h)
            put("ses", c, f["name"], h, L.reindex(UNI[(c, f["name"], h)].index))

# 5c pooled ridge-AR and pooled gradient boosting. Target = log-ratio to persistence (residual learning)
def fit_predict_ml(kind, f, h):
    B = BIG[h]; Xc = B[ML_COLS]
    trm = (B["date"] < f["cal_start"]) & B["y"].notna() & B["l0"].notna()
    ty = np.log1p(B.loc[trm, "y"]) - B.loc[trm, "l0"]
    if kind == "hgb":
        m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, min_samples_leaf=30,
                                          l2_regularization=1.0, random_state=CFG["SEED"])
    else:
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0))
    m.fit(Xc[trm], ty)
    return m
for f in FOLDS:
    for h in HS:
        for kind in ("ridge", "hgb"):
            m = fit_predict_ml(kind, f, h)
            for c in CITIES:
                if not VALID[(c, f["name"])]: continue
                U = UNI[(c, f["name"], h)]
                B = BIG[h]; sub = B[(B.city == c)].set_index("date").loc[U.index]
                d = m.predict(sub[ML_COLS])
                put(kind, c, f["name"], h, pd.Series(np.clip(np.expm1(sub["l0"].to_numpy() + d), 0, None), index=U.index))
    log("ridge/hgb done for fold", f["name"])


## Section VII: Deep Sequence Modeling — Pooled Multi-Horizon LSTM

To model non-linear temporal dependencies across heterogeneous urban environments, we implement a **pooled multi-horizon recurrent neural network** in PyTorch.

### Network Architecture:
1. **Input Sequence:** Lookback window of length $L = 60$ historical daily steps. Each time step $\tau$ contains a 4-dimensional feature vector:
   $$\mathbf{u}_\tau = [\tilde{y}_\tau, m_\tau, a_\tau, \text{miss}_{30}(\tau)]^T$$
   where $\tilde{y}_\tau$ is per-city standardized log-concentration, $m_\tau$ is the missingness mask, and $a_\tau$ is observation age.
2. **Recurrent Encoder:** A long short-term memory (LSTM) recurrent layer with hidden dimension $H = 64$:
   $$\mathbf{h}_\tau, \mathbf{c}_\tau = \text{LSTM}(\mathbf{u}_\tau, \mathbf{h}_{\tau-1}, \mathbf{c}_{\tau-1})$$
3. **Spatial Entity Embedding:** A trainable lookup table mapping each discrete city index $c \in \{1, \dots, N_C\}$ to a low-dimensional spatial vector $\mathbf{e}_c \in \mathbb{R}^4$.
4. **Multi-Horizon Direct Decoder:** Final recurrent hidden state $\mathbf{h}_L$ is concatenated with $\mathbf{e}_c$ and passed through a multi-layer perceptron:
   $$\hat{\mathbf{z}} = \text{MLP}([\mathbf{h}_L \,;\, \mathbf{e}_c]) \in \mathbb{R}^{|\mathcal{H}|}$$
   directly outputting residual forecasts for all horizons $h \in \{1, 3, 7\}$ simultaneously.
5. **Loss Function & Training:** Trained with masked Smooth L1 (Huber) loss and early stopping evaluated against the validation calibration partition.

In [ ]:
# %% [6] LSTM (PyTorch, pooled over cities, direct multi-horizon, residual to last observation)
def run_lstm():
    import torch.nn as nn
    P = CFG["LSTM"]; L = P["L"]; dev = DEVICE
    class Net(nn.Module):
        def __init__(s):
            super().__init__(); s.emb = nn.Embedding(NC, 4)
            s.rnn = nn.LSTM(4, P["hidden"], batch_first=True)
            s.head = nn.Sequential(nn.Linear(P["hidden"] + 4, 64), nn.ReLU(), nn.Linear(64, len(HS)))
        def forward(s, x, cid):
            o, _ = s.rnn(x); return s.head(torch.cat([o[:, -1], s.emb(cid)], 1))
    for f in FOLDS:
        # per-city normalisation from TRAIN period only
        norm = {}; arr = {}
        for c in CITIES:
            y = PANEL[c]["PM2.5"]; ly = np.log1p(y); tr = ly[ly.index < f["cal_start"]].dropna()
            mu, sd = (tr.mean(), tr.std() + 1e-6) if len(tr) > 30 else (ly.mean(), ly.std() + 1e-6)
            z = ((ly - mu) / sd).to_numpy(); mask = (~np.isnan(z)).astype(np.float32)
            zl = pd.Series(z).ffill().to_numpy(); doy = PANEL[c].index.dayofyear.to_numpy()
            arr[c] = dict(z=np.nan_to_num(z, nan=0.0).astype(np.float32), m=mask, zl=zl, zraw=z,
                          s=np.sin(2*np.pi*doy/365.25).astype(np.float32), co=np.cos(2*np.pi*doy/365.25).astype(np.float32),
                          dates=PANEL[c].index)
            norm[c] = (mu, sd)
        def windows(c, i0, i1):
            a = arr[c]; ii = np.arange(max(i0, L - 1), i1)
            X = np.stack([np.stack([a["z"][i-L+1:i+1], a["m"][i-L+1:i+1], a["s"][i-L+1:i+1], a["co"][i-L+1:i+1]], 1) for i in ii]) if len(ii) else np.zeros((0, L, 4), np.float32)
            return ii, X.astype(np.float32)
        Xtr, Ctr, Btr, Ytr, Mtr, Xva, Cva, Bva, Yva, Mva = [], [], [], [], [], [], [], [], [], []
        for c in CITIES:
            a = arr[c]; n = len(a["z"]); dates = a["dates"]
            ii, X = windows(c, 0, n)
            for k, i in enumerate(ii):
                if np.isnan(a["zl"][i]): continue
                ys, ms = [], []
                for h in HS:
                    j = i + h
                    ok = (j < n) and (dates[j] < f["cal_start"]) and (a["m"][j] == 1.0)
                    ys.append(a["zraw"][j] if ok else 0.0); ms.append(1.0 if ok else 0.0)
                if sum(ms) == 0: continue
                tgt = (Xva if dates[i] >= f["cal_start"] - pd.Timedelta(days=P["val_days"] + HMAX) else Xtr)
                if tgt is Xva: Xva.append(X[k]); Cva.append(CIDX[c]); Bva.append(a["zl"][i]); Yva.append(ys); Mva.append(ms)
                else: Xtr.append(X[k]); Ctr.append(CIDX[c]); Btr.append(a["zl"][i]); Ytr.append(ys); Mtr.append(ms)
        tt = lambda v, dt=torch.float32: torch.tensor(np.asarray(v), dtype=dt, device=dev)
        Xtr, Ctr, Btr, Ytr, Mtr = tt(Xtr), tt(Ctr, torch.long), tt(Btr), tt(Ytr), tt(Mtr)
        Xva, Cva, Bva, Yva, Mva = tt(Xva), tt(Cva, torch.long), tt(Bva), tt(Yva), tt(Mva)
        def lossf(net, X, C, B, Y, M):
            p = B[:, None] + net(X, C); l = nn.functional.smooth_l1_loss(p, Y, reduction="none")
            return (l * M).sum() / M.sum().clamp(min=1)
        # inference windows for every origin, once per fold
        inf = {}
        for c in CITIES:
            ii, X = windows(c, 0, len(arr[c]["z"])); inf[c] = (ii, tt(X), tt([CIDX[c]] * len(ii), torch.long), tt(arr[c]["zl"][ii]))
        acc = {c: np.zeros((len(inf[c][0]), len(HS))) for c in CITIES}
        for seed in P["seeds"]:
            torch.manual_seed(seed); np.random.seed(seed)
            net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), lr=P["lr"])
            best, best_state, bad = 1e18, None, 0
            for ep in range(P["epochs"]):
                net.train(); perm = torch.randperm(len(Xtr), device=dev)
                for s in range(0, len(perm), P["bs"]):
                    b = perm[s:s+P["bs"]]; opt.zero_grad(); lossf(net, Xtr[b], Ctr[b], Btr[b], Ytr[b], Mtr[b]).backward(); opt.step()
                net.eval()
                with torch.no_grad(): v = lossf(net, Xva, Cva, Bva, Yva, Mva).item() if len(Xva) else 0.0
                if v < best - 1e-5: best, bad = v, 0; best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
                else:
                    bad += 1
                    if bad >= P["patience"]: break
            net.load_state_dict(best_state); net.eval()
            with torch.no_grad():
                for c in CITIES:
                    ii, X, C, B = inf[c]; acc[c] += (B[:, None] + net(X, C)).cpu().numpy()
        for c in CITIES:
            if not VALID[(c, f["name"])]: continue
            ii = inf[c][0]; zhat = acc[c] / len(P["seeds"]); mu, sd = norm[c]; dates = arr[c]["dates"]
            for k, h in enumerate(HS):
                tgt_idx = ii + h; okk = tgt_idx < len(dates)
                s = pd.Series(np.clip(np.expm1(zhat[okk, k] * sd + mu), 0, None), index=dates[tgt_idx[okk]])
                put("lstm", c, f["name"], h, s.reindex(UNI[(c, f["name"], h)].index))
        log("lstm done for fold", f["name"])
LSTM_OK = False
if CFG["LSTM"]["enabled"] and torch is not None:
    try: run_lstm(); LSTM_OK = True
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"!! LSTM FAILED and is skipped: {type(e).__name__}: {e}"); PRED.pop("lstm", None)
else: print("LSTM skipped (disabled or torch missing)")


## Section VIII: Zero-Shot Time-Series Foundation Models (TSFMs)

We benchmark state-of-the-art pretrained foundation models developed for time-series forecasting:

### Evaluated Model Variants:
* **Chronos-Bolt-Small & Chronos-Bolt-Base (Amazon):** Patch-based encoder-decoder transformer architectures trained on billions of diverse time-series observations. Chronos-Bolt maps continuous patches directly to output quantile distributions, drastically reducing inference latency relative to tokenized autoregressive models.
* **Chronos-2 (Amazon):** Next-generation foundation architecture with enhanced scaling parameters and continuous representation mechanisms.

### Operational Zero-Shot Protocol:
1. **No In-Domain Fine-Tuning:** Weights remain frozen; models are evaluated purely on zero-shot inference capability.
2. **Strict Context Window Isolation:** For each forecast origin date $t$, the input context is strictly restricted to historical values up to $t$.
3. **Probabilistic Quantiles:** The models directly emit distribution quantiles $q_{0.10}, q_{0.50},$ and $q_{0.90}$ over prediction horizon $h$.

In [ ]:
# %% [7] Foundation models (zero-shot). Context = ONLY data up to the forecast origin (no leakage).
TSFM_IDS = {"chronos-bolt-small": "amazon/chronos-bolt-small", "chronos-bolt-base": "amazon/chronos-bolt-base", "chronos-2": "amazon/chronos-2"}
QL = [0.1, 0.5, 0.9]
def to_np(t): return t.float().cpu().numpy() if hasattr(t, "float") else np.asarray(t)
class MockPipe:                                   # plumbing test only
    def predict_quantiles(self, inputs, prediction_length, quantile_levels, **kw):
        out = []
        for x in inputs:
            v = np.asarray(x, float); v = v[np.isfinite(v)]; m = v[-1] if len(v) else 0.0
            out.append(np.stack([np.full(prediction_length, m*0.7), np.full(prediction_length, m), np.full(prediction_length, m*1.4)], 1))
        return np.stack(out).astype(np.float32), None

def origins_for(c):
    lo = min(f["cal_start"] for f in FOLDS) - pd.Timedelta(days=HMAX); hi = max(f["test_end"] for f in FOLDS) - pd.Timedelta(days=1)
    return PANEL[c].index[(PANEL[c].index >= lo) & (PANEL[c].index <= hi)]

def run_chronos(name):
    cache = f"{CFG['OUT']}/cache_{name}.pkl"
    if os.path.exists(cache): log("cache hit", name); return pickle.load(open(cache, "rb"))
    if CFG["MOCK_TSFM"]: pipe = MockPipe()
    else:
        from chronos import BaseChronosPipeline
        pipe = BaseChronosPipeline.from_pretrained(TSFM_IDS[name], device_map=DEVICE)
    res = {}
    for c in CITIES:
        y = PANEL[c]["PM2.5"].to_numpy(np.float32); pos = {d: i for i, d in enumerate(PANEL[c].index)}
        org = origins_for(c); ctxs, keep = [], []
        for d in org:
            i = pos[d]; ctx = y[max(0, i - CFG["CTX"] + 1): i + 1]
            if np.isfinite(ctx).sum() >= CFG["MIN_CTX_OBS"]: ctxs.append(ctx); keep.append(d)
        Q = np.full((len(org), HMAX, 3), np.nan, np.float32); kpos = {d: k for k, d in enumerate(org)}
        for s in range(0, len(ctxs), CFG["TSFM_BATCH"]):
            batch = ctxs[s:s+CFG["TSFM_BATCH"]] if CFG["MOCK_TSFM"] else [torch.tensor(x) for x in ctxs[s:s+CFG["TSFM_BATCH"]]]
            q, _ = pipe.predict_quantiles(inputs=batch, prediction_length=HMAX, quantile_levels=QL)
            q = np.stack([to_np(t)[0] for t in q]) if isinstance(q, (list, tuple)) else to_np(q)   # Chronos-2 returns a list
            for j, d in enumerate(keep[s:s+CFG["TSFM_BATCH"]]): Q[kpos[d]] = q[j]
        res[c] = pd.DataFrame(Q.reshape(len(org), -1), index=org)
        log(name, c, "done", len(ctxs), "contexts")
    pickle.dump(res, open(cache, "wb")); return res

def run_timesfm():                                 # OPTIONAL / UNTESTED: TimesFM 3 (pip install "timesfm[torch]")
    from timesfm3 import TimesFM3Forecaster
    fc = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch", device=DEVICE); res = {}
    for c in CITIES:
        y = PANEL[c]["PM2.5"].to_numpy(np.float32); pos = {d: i for i, d in enumerate(PANEL[c].index)}
        org = origins_for(c); Q = np.full((len(org), HMAX, 3), np.nan, np.float32)
        for k, d in enumerate(org):
            ctx = y[max(0, pos[d] - CFG["CTX"] + 1): pos[d] + 1]
            if np.isfinite(ctx).sum() < CFG["MIN_CTX_OBS"]: continue
            o = fc.predict(ctx, horizon=HMAX, return_quantiles=True)
            Q[k, :, 1] = o.forecast[:HMAX]
            qs = o.quantiles
            if qs is not None and qs.ndim == 2:
                if qs.shape[-1] == 10: qs = qs[:, 1:]
                if qs.shape[-1] == 9: Q[k, :, 0], Q[k, :, 2] = qs[:HMAX, 0], qs[:HMAX, 8]
        res[c] = pd.DataFrame(Q.reshape(len(org), -1), index=org)
    return res

TSFM_OK, NATIVE = [], {}
runners = {n: (lambda n=n: run_chronos(n)) for n in CFG["TSFMS"]}
if CFG["RUN_TIMESFM"]: runners["timesfm3"] = run_timesfm
for name, fn in runners.items():
    try:
        out = fn()
        for (c, fname, h), U in UNI.items():
            D = out[c]; tdates = U.index; odates = tdates - pd.Timedelta(days=h)
            sub = D.reindex(odates).to_numpy().reshape(len(odates), HMAX, 3)[:, h - 1, :]
            q10, q50, q90 = (pd.Series(sub[:, k], index=tdates) for k in range(3))
            put(name, c, fname, h, q50.clip(lower=0)); NATIVE[(name, c, fname, h)] = (q10.clip(lower=0), q90.clip(lower=0))
        TSFM_OK.append(name)
    except Exception as e:
        print(f"!! {name} FAILED and is skipped: {type(e).__name__}: {e}")
log("foundation models that ran:", TSFM_OK)


## Section IX: Common Evaluation Universe & Distribution-Free Conformal Prediction

### Common Universe Assembly
For each combination of city $c$, test fold $f$, and horizon $h$, we join predictions across all models. An evaluation record is retained if and only if **every candidate model** produces a non-null forecast for that target date.

### Conformal Prediction Intervals
To provide rigorous, distribution-free uncertainty guarantees, we implement **Split Conformal Prediction**. Let $\mathcal{D}_{\text{cal}} = \{(x_i, y_i)\}_{i=1}^n$ be the held-out calibration set.
1. **Non-Conformity Score:** Absolute residual $R_i = |y_i - \hat{y}_i|$.
2. **Empirical Quantile:** For miscoverage budget $\alpha \in \{0.10, 0.20\}$, compute the adjusted quantile:
   $$\hat{q}_{1-\alpha} = \text{Quantile}\left(\left\{R_i\right\}_{i=1}^n, \frac{\lceil (n+1)(1-\alpha) \rceil}{n}\right)$$
3. **Prediction Interval:**
   $$\mathcal{C}_{1-\alpha}(x_{n+1}) = \left[\hat{y}_{n+1} - \hat{q}_{1-\alpha}, \; \hat{y}_{n+1} + \hat{q}_{1-\alpha}\right]$$

### Evaluated Calibration Schemes:
* **Global Conformal:** Pooled across all calibration days.
* **Seasonal Conformal:** Partitioned into Winter ($Nov–Feb$) vs Non-Winter strata to account for seasonal heteroskedasticity.
* **Rolling Conformal:** Calibrated on a trailing window of $N=365$ days immediately preceding the forecast date.
* **Scaled Conformal:** Normalized non-conformity score $R_i / s_i$ using local rolling volatility.
* **TSFM Native Quantiles:** Model-derived $80\%$ prediction interval $[q_{0.10}, q_{0.90}]$.

In [ ]:
# %% [8] Assemble evaluation table on the COMMON row set + conformal intervals
MODELS = [m for m in ["persist", "ma7", "clim", "ses", "ridge", "hgb", "lstm"] + TSFM_OK if m in PRED]
rows = []
for (c, fname, h), U in UNI.items():
    D = pd.DataFrame({m: PRED[m][(c, fname, h)].reindex(U.index) for m in MODELS})
    ok = D.notna().all(axis=1)                                   # every model must have a forecast for the row
    Uo = U[ok]; 
    for m in MODELS:
        d = pd.DataFrame({"model": m, "city": c, "fold": fname, "h": h, "date": Uo.index, "split": Uo["split"].to_numpy(),
                          "y": Uo["y"].to_numpy(), "pred": D.loc[ok, m].to_numpy(), "scale": U.attrs["scale"],
                          "spike": (np.expm1(Uo["l0"].to_numpy()) > U.attrs["p90"]), "p90": U.attrs["p90"], "age": Uo["age"].to_numpy(), "miss30": Uo["miss30"].to_numpy()})
        if (m, c, fname, h) in NATIVE:
            q10, q90 = NATIVE[(m, c, fname, h)]; d["q10"] = q10.reindex(Uo.index).to_numpy(); d["q90"] = q90.reindex(Uo.index).to_numpy()
        else: d["q10"] = np.nan; d["q90"] = np.nan
        rows.append(d)
P = pd.concat(rows, ignore_index=True); P["regime"] = regime_of(P["date"])
P["ae"] = (P.y - P.pred).abs(); P["mase"] = P.ae / P.scale
log("common-set rows:", len(P) // len(MODELS), "per model")

def cq(r, alpha):
    n = len(r); k = min(int(math.ceil((n + 1) * (1 - alpha))), n); return np.sort(r)[k - 1]

def conformal_group(g, alpha):
    g = g.sort_values("date"); n = len(g); h = int(g.h.iloc[0])
    dd = g.date.to_numpy().astype("datetime64[D]").astype(np.int64); y = g.y.to_numpy(); p = g.pred.to_numpy()
    res = np.abs(y - p); cal = (g.split == "calib").to_numpy(); test = ~cal
    win = ((g.date.dt.month >= 11) | (g.date.dt.month <= 2)).to_numpy()
    if cal.sum() < CFG["MIN_STRATUM"] or test.sum() == 0: return None
    qg = cq(res[cal], alpha)
    qw = cq(res[cal & win], alpha) if (cal & win).sum() >= CFG["MIN_STRATUM"] else qg
    qn = cq(res[cal & ~win], alpha) if (cal & ~win).sum() >= CFG["MIN_STRATUM"] else qg
    qs = np.where(win, qw, qn); qr = np.full(n, np.nan)
    for j in np.where(test)[0]:
        m = (dd <= dd[j] - h) & (dd > dd[j] - h - CFG["ROLL_WIN"])          # only residuals already realised at forecast time
        qr[j] = cq(res[m], alpha) if m.sum() >= CFG["MIN_STRATUM"] else qg
    out = []
    for meth, q in (("global", np.full(n, qg)), ("seasonal", qs), ("rolling", qr)):
        out.append((meth, np.maximum(p - q, 0), p + q))
    sc = np.maximum(p, CFG["SCALE_FLOOR"])                          # level-normalised ("locally adaptive") conformal:
    qsc = cq((res / sc)[cal], alpha)                                # error grows with the pollution level, so scale by forecast
    out.append(("scaled", np.maximum(p - qsc * sc, 0), p + qsc * sc))
    if np.isfinite(g.q10.to_numpy()).any() and abs(alpha - 0.2) < 1e-9: out.append(("native", g.q10.to_numpy(), g.q90.to_numpy()))
    recs = []
    for meth, lo, hi in out:
        cv = ((y >= lo) & (y <= hi)).astype(np.float32); w = hi - lo
        wk = w + (2 / alpha) * np.clip(lo - y, 0, None) + (2 / alpha) * np.clip(y - hi, 0, None)
        d = g.loc[:, ["model", "city", "fold", "h", "date", "regime", "spike"]].copy()
        d["method"], d["alpha"], d["cov"], d["width"], d["wink"], d["lo"], d["hi"] = meth, alpha, cv, w, wk, lo, hi
        recs.append(d[test])
    return pd.concat(recs)

parts = []
for (m, c, fname, h), g_ in P.groupby(["model", "city", "fold", "h"]):
    for a in CFG["ALPHAS"]:
        r = conformal_group(g_, a)
        if r is not None: parts.append(r)
I = pd.concat(parts, ignore_index=True)
E = P[P.split == "test"].copy()
log("evaluation rows:", len(E) // len(MODELS), "test rows per model | interval rows:", len(I))


## Section X: Statistical Inference & Hypothesis Testing Framework

To ensure publication-standard rigor, we adopt formal econometric and statistical testing:

### 1. Point Forecasting Metrics:
* **Mean Absolute Scaled Error (MASE):**
  $$\text{MASE} = \frac{\frac{1}{N} \sum_{t=1}^N |y_t - \hat{y}_t|}{\frac{1}{N_{\text{train}}-1} \sum_{i=2}^{N_{\text{train}}} |y_i - y_{i-1}|}$$
* **Mean Absolute Error (MAE) & Root Mean Squared Error (RMSE)**.
* **Forecast Skill Score:** Relative improvement over persistence:
  $$\text{Skill} = 1 - \frac{\text{MAE}_{\text{model}}}{\text{MAE}_{\text{persistence}}}$$

### 2. Extreme Event Detection ($y > P_{90}$):
Air quality alerts are triggered when concentrations exceed the 90th percentile of historical training data ($P_{90}$). We evaluate:
$$\text{Recall} = \frac{TP}{TP + FN}, \quad \text{Precision} = \frac{TP}{TP + FP}, \quad F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

### 3. Spatial Cluster Bootstrapping:
Because errors across proximate Indian cities exhibit spatial autocorrelation, ordinary i.i.d. bootstrap underestimates variance. We apply **Cluster Bootstrapping** resampling cities with replacement ($B = 1000$) to construct valid 95% confidence intervals.

### 4. Diebold-Mariano Predictive Superiority Tests:
For each city and horizon $h$, we conduct the Diebold-Mariano test under absolute error loss differential $d_t = |e_{1,t}| - |e_{2,t}|$, applying the Harvey-Leybourne-Newbold (HLN) small-sample modification and HAC variance estimation with lag $h-1$.
* **Multiple Testing Correction:** Family-wise error rate across candidate models is controlled using the **Holm-Bonferroni step-down procedure**.
* **Pooled Meta-Analysis (Stouffer's Z-Score):** To evaluate overall performance across all $K$ cities simultaneously:
  $$Z_{\text{pooled}} = \frac{1}{\sqrt{K}} \sum_{k=1}^K \Phi^{-1}(1 - p_k), \quad p_{\text{pooled}} = 1 - \Phi(Z_{\text{pooled}})$$

In [ ]:
# %% [9] Statistics: macro over cities, cluster bootstrap over cities, Diebold-Mariano, coverage by regime
from scipy import stats as sst
def cboot(v, B=CFG["BOOT"], seed=0):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) == 0: return np.nan, np.nan, np.nan
    if len(v) == 1: return v[0], np.nan, np.nan
    rng = np.random.default_rng(seed); m = v[rng.integers(0, len(v), (B, len(v)))].mean(1)
    return v.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

# 9a point accuracy
pt = []
for (m, h), g_ in E.groupby(["model", "h"]):
    pc = g_.groupby("city").agg(mase=("mase", "mean"), mae=("ae", "mean"), rmse=("ae", lambda x: np.sqrt((x**2).mean())))
    mm, lo, hi = cboot(pc.mase)
    pt.append(dict(model=m, h=h, MASE=mm, MASE_lo=lo, MASE_hi=hi, MAE=pc.mae.mean(), RMSE=pc.rmse.mean(), n=len(g_)))
PT = pd.DataFrame(pt)
base = E[E.model == "persist"].groupby(["h", "city"]).ae.mean().rename("ae_p")
sk = E.groupby(["model", "h", "city"]).ae.mean().rename("ae_m").reset_index().merge(base.reset_index(), on=["h", "city"])
sk["skill"] = 1 - sk.ae_m / sk.ae_p
SK = sk.groupby(["model", "h"]).skill.apply(lambda v: pd.Series(cboot(v), index=["skill", "skill_lo", "skill_hi"])).unstack().reset_index()
PT = PT.merge(SK, on=["model", "h"])
def ev_metrics(g_):
    t = (g_.y > g_.p90).to_numpy(); p = (g_.pred > g_.p90).to_numpy()
    tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
    rec = tp / (tp + fn) if tp + fn else np.nan; pre = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * pre * rec / (pre + rec) if (pre == pre and rec == rec and pre + rec > 0) else np.nan
    return pd.Series(dict(ev_recall=rec, ev_prec=pre, ev_f1=f1, n_pos=int(t.sum()), n_days=len(t)))
EV_CITY = E.groupby(["model", "h", "city"]).apply(ev_metrics).reset_index()
EV = EV_CITY.groupby(["model", "h"]).agg(ev_recall=("ev_recall", "mean"), ev_prec=("ev_prec", "mean"),
      ev_f1=("ev_f1", "mean"), n_pos=("n_pos", "sum"), n_days=("n_days", "sum")).reset_index()
# city-level bootstrap CIs for the three event-detection metrics, same construction as every other CI in this script
evci = []
for (m, h), g_ in EV_CITY.groupby(["model", "h"]):
    row = {"model": m, "h": h}
    for met in ("ev_recall", "ev_prec", "ev_f1"):
        mm, lo, hi = cboot(g_[met])
        row[f"{met}_lo"], row[f"{met}_hi"] = lo, hi
    evci.append(row)
EV = EV.merge(pd.DataFrame(evci), on=["model", "h"])
PT = PT.merge(EV, on=["model", "h"])
# same breakdown but ALSO split by fold, so the lockdown fold's diluted positive rate is visible rather than pooled away
def ev_metrics_n(g_):
    t = (g_.y > g_.p90).to_numpy(); p = (g_.pred > g_.p90).to_numpy()
    tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
    rec = tp / (tp + fn) if tp + fn else np.nan; pre = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * pre * rec / (pre + rec) if (pre == pre and rec == rec and pre + rec > 0) else np.nan
    return pd.Series(dict(ev_recall=rec, ev_prec=pre, ev_f1=f1, n_pos=int(t.sum()), n_days=len(t)))
EV_FOLD_CITY = E.groupby(["model", "h", "fold", "city"]).apply(ev_metrics_n).reset_index()
EV_FOLD = EV_FOLD_CITY.groupby(["model", "h", "fold"]).agg(ev_recall=("ev_recall", "mean"), ev_prec=("ev_prec", "mean"),
      ev_f1=("ev_f1", "mean"), n_pos=("n_pos", "sum"), n_days=("n_days", "sum")).reset_index()

# 9b Diebold-Mariano vs persistence (per city, absolute-error loss, HAC lag h-1, Harvey correction), Holm over model x city per horizon
def dm(d, h):
    d = np.asarray(d, float); n = len(d)
    if n < 30: return np.nan, np.nan
    db = d.mean(); gam = lambda k: np.sum((d[k:] - db) * (d[:n - k] - db)) / n
    var = gam(0) + 2 * sum(gam(k) for k in range(1, h)); var = var if var > 0 else gam(0)
    if var <= 0: return np.nan, np.nan
    s = db / np.sqrt(var / n) * math.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return s, 2 * (1 - sst.t.cdf(abs(s), n - 1))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(np.where(np.isnan(p), 2, p)); adj = np.full(len(p), np.nan); prev = 0
    for r, i in enumerate(o):
        if np.isnan(p[i]): continue
        prev = max(prev, min(1, p[i] * (len(p) - r))); adj[i] = prev
    return adj
dmr = []
for h in HS:
    ref = E[(E.model == "persist") & (E.h == h)].set_index(["city", "date"]).ae
    for m in [x for x in MODELS if x != "persist"]:
        cur = E[(E.model == m) & (E.h == h)].set_index(["city", "date"]).ae
        for c in CITIES:
            if c not in cur.index.get_level_values(0): continue
            j = pd.concat([cur.loc[c], ref.loc[c]], axis=1, keys=["m", "p"]).dropna().sort_index()
            s, p = dm((j.m - j.p).to_numpy(), h); dmr.append(dict(h=h, model=m, city=c, dm=s, p=p, mean_diff=(j.m - j.p).mean()))
DM = pd.DataFrame(dmr)
DM["p_holm"] = DM.groupby("h").p.transform(lambda s: pd.Series(holm(s.to_numpy()), index=s.index))
def stouffer(pvals, signs):
    '''Pool per-city DM tests into one statistic per (model,h) with Stouffer's method, sign-oriented so the
    combined test is one-sided "model beats persistence"; drops any city with a NaN test (too few residual days).'''
    p = np.asarray(pvals, float); sgn = np.asarray(signs, float); ok = ~np.isnan(p)
    p, sgn = p[ok], sgn[ok]
    if len(p) == 0: return np.nan, np.nan, 0
    p = np.clip(p, 1e-300, 1 - 1e-16)
    z_two_sided = sst.norm.isf(p / 2)                      # two-sided per-city DM p -> per-city |z|
    z_signed = z_two_sided * (-np.sign(sgn))                # negative mean_diff (model beats persistence) -> positive z
    Z = z_signed.sum() / math.sqrt(len(z_signed))
    return Z, sst.norm.sf(Z), len(p)                        # one-sided pooled p: is the model better than persistence, pooled over cities
DMS = DM.groupby(["h", "model"]).apply(lambda g_: pd.Series(dict(cities_sig_better=int(((g_.p_holm < .05) & (g_.mean_diff < 0)).sum()),
      cities_sig_worse=int(((g_.p_holm < .05) & (g_.mean_diff > 0)).sum()), n_cities=len(g_),
      **dict(zip(["stouffer_z", "stouffer_p", "stouffer_n_cities"], stouffer(g_.p.to_numpy(), g_.mean_diff.to_numpy())))))).reset_index()

# 9c coverage / width / interval score by stratum
STR = {"all": lambda d: np.ones(len(d), bool), "winter": lambda d: (d.regime == "winter").to_numpy(), "other": lambda d: (d.regime == "other").to_numpy(),
       "lockdown": lambda d: (d.regime == "lockdown").to_numpy(), "high_state": lambda d: d.spike.to_numpy()}
cv = []
cv_fold = []
for (m, h, a, meth), g_ in I.groupby(["model", "h", "alpha", "method"]):
    for sname, fn in STR.items():
        s = g_[fn(g_)]
        if len(s) < 20: continue
        pc = s.groupby("city").agg(cov=("cov", "mean"), w=("width", "mean"), wk=("wink", "mean"), n=("cov", "size"))
        mm, lo, hi = cboot(pc["cov"])
        cv.append(dict(model=m, h=h, alpha=a, method=meth, stratum=sname, cov=mm, cov_lo=lo, cov_hi=hi, gap=mm - (1 - a), width=pc.w.mean(), wink=pc.wk.mean(), n=int(pc.n.sum()), cities=len(pc)))
        # same cell, but split by fold as well, so the lockdown fold cannot silently dominate a pooled number
        for fld, gf in s.groupby("fold"):
            pcf = gf.groupby("city").agg(cov=("cov", "mean"), w=("width", "mean"), n=("cov", "size"))
            if len(pcf) < 2: continue
            mmf, lof, hif = cboot(pcf["cov"])
            cv_fold.append(dict(model=m, h=h, alpha=a, method=meth, stratum=sname, fold=fld, cov=mmf, cov_lo=lof, cov_hi=hif,
                                 width=pcf.w.mean(), n=int(pcf.n.sum()), cities=len(pcf)))
CV = pd.DataFrame(cv)
CV_FOLD = pd.DataFrame(cv_fold)

# 9d missing-data buckets (at the forecast ORIGIN)
E["miss_bucket"] = pd.cut(E.miss30, [-1e-9, 0, 0.25, 1.0], labels=["none", "1-25%", ">25%"])
E["age_bucket"] = pd.cut(E.age, [-1, 0, 7, 1e9], labels=["observed today", "1-7 d stale", ">7 d stale"])
MS = []
for kind in ("miss_bucket", "age_bucket"):
    for (m, h, b), g_ in E.groupby(["model", "h", kind], observed=True):
        pc = g_.groupby("city").mase.mean(); mm, lo, hi = cboot(pc)
        MS.append(dict(kind=kind, model=m, h=h, bucket=str(b), MASE=mm, lo=lo, hi=hi, n=len(g_), cities=len(pc)))
MS = pd.DataFrame(MS)
for nme, dfx in dict(point_metrics=PT, dm_tests=DM, dm_summary=DMS, coverage=CV, coverage_by_fold=CV_FOLD, missingness=MS,
                    event_detection_by_fold=EV_FOLD, event_detection_by_city=EV_CITY).items():
    dfx.to_csv(f"{CFG['OUT']}/{nme}.csv", index=False)
E.to_csv(f"{CFG['OUT']}/test_rows_point.csv.gz", index=False)
I.to_csv(f"{CFG['OUT']}/interval_rows.csv.gz", index=False)   # raw interval-level rows, needed for any future Table 4 CI / fold recomputation
log("tables saved")


## Section XI: Empirical Visualizations & Diagnostic Analysis

This section generates high-resolution, publication-ready figures:
1. **Figure 1 (MASE Multi-Horizon Benchmark):** Comparative bar charts displaying macro-averaged MASE across horizons $h \in \{1, 3, 7\}$ days with 95% cluster-bootstrapped error bars.
2. **Figure 2 (Conformal Interval Reliability):** Stratified empirical coverage versus nominal coverage levels ($80\%$ and $90\%$) across environmental regimes (All Days, Winter Smog, COVID Lockdown, High-State).
3. **Figure 3 (Exemplary Trajectories & Uncertainty Envelopes):** Time-series plots of ground-truth $\text{PM}_{2.5}$ dynamics alongside model forecasts and conformal uncertainty bands during extreme pollution events.

In [ ]:
# %% [10] Figures
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
MCOL = {m: plt.cm.tab10(i) for i, m in enumerate(MODELS)}
fig, ax = plt.subplots(1, len(HS), figsize=(4.2 * len(HS), 3.6), sharey=True)
for a_, h in zip(np.atleast_1d(ax), HS):
    s = PT[PT.h == h].set_index("model").loc[MODELS]
    a_.bar(range(len(s)), s.MASE, yerr=[s.MASE - s.MASE_lo, s.MASE_hi - s.MASE], color=[MCOL[m] for m in s.index], capsize=2)
    a_.set_xticks(range(len(s))); a_.set_xticklabels(s.index, rotation=60, ha="right", fontsize=7); a_.set_title(f"h = {h} day(s)"); a_.axhline(1, ls=":", c="k", lw=.7)
np.atleast_1d(ax)[0].set_ylabel("MASE (mean over cities, 95% CI over cities)"); plt.tight_layout(); plt.savefig(f"{CFG['OUT']}/fig1_mase.png", dpi=200); plt.close()

show = [m for m in ["hgb", "lstm"] + TSFM_OK[:2] if m in MODELS]; strata = ["all", "winter", "other", "lockdown", "high_state"]; meths = ["native", "global", "seasonal", "rolling", "scaled"]
for a in CFG["ALPHAS"]:
    fig, axs = plt.subplots(len(show), 2, figsize=(10, 2.6 * len(show)), squeeze=False)
    for r, m in enumerate(show):
        for cidx, h in enumerate([HS[0], HS[-1]]):
            ax_ = axs[r][cidx]; w = 0.16
            for k, me in enumerate(meths):
                s = CV[(CV.model == m) & (CV.h == h) & (CV.alpha == a) & (CV.method == me)].set_index("stratum").reindex(strata)
                if s["cov"].notna().any(): ax_.bar(np.arange(len(strata)) + (k - 2) * w, s["cov"], w, yerr=[(s["cov"] - s.cov_lo).fillna(0), (s.cov_hi - s["cov"]).fillna(0)], label=me, capsize=1.5)
            ax_.axhline(1 - a, c="k", ls="--", lw=.8); ax_.set_ylim(0.4, 1.02); ax_.set_xticks(range(len(strata))); ax_.set_xticklabels(strata, fontsize=8)
            ax_.set_title(f"{m}, h={h}", fontsize=9)
            if r == 0 and cidx == 0: ax_.legend(fontsize=7, ncol=5)
    plt.suptitle(f"Empirical coverage by regime (nominal {int((1-a)*100)}%)"); plt.tight_layout(); plt.savefig(f"{CFG['OUT']}/fig2_coverage_alpha{a}.png", dpi=200); plt.close()

best_tsfm = min(TSFM_OK, key=lambda m: PT[(PT.model == m) & (PT.h == HS[0])].MASE.iloc[0]) if TSFM_OK else "hgb"
def plot_example(city, model, h, start, end, alpha=0.2, method="seasonal"):
    d = I[(I.city == city) & (I.model == model) & (I.h == h) & (I.alpha == alpha) & (I.method == method) & (I.date >= start) & (I.date <= end)].sort_values("date")
    yy = E[(E.city == city) & (E.model == model) & (E.h == h) & (E.date >= start) & (E.date <= end)].sort_values("date")
    if len(d) == 0 or len(yy) == 0: return
    fig, ax = plt.subplots(figsize=(9, 3.2)); ax.plot(yy.date, yy.y, "k", lw=1, label="observed"); ax.plot(yy.date, yy.pred, lw=1, label=f"{model} forecast")
    ax.fill_between(d.date, d.lo.to_numpy(), d.hi.to_numpy(), alpha=.25, label=f"{int((1-alpha)*100)}% {method} conformal")
    ax.set_title(f"{city}, h={h}"); ax.legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{CFG['OUT']}/fig3_example_{city}_{model}_h{h}.png", dpi=200); plt.close()
for c_ in ("Delhi", "Lucknow"):
    if c_ in CITIES:
        plot_example(c_, best_tsfm, 1, "2019-11-01", "2020-02-28"); plot_example(c_, best_tsfm, 7, "2019-11-01", "2020-02-28")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for a_, kind in zip(ax, ("miss_bucket", "age_bucket")):
    s = MS[(MS.kind == kind) & (MS.h == HS[0]) & (MS.model.isin(["persist", "hgb", "lstm"] + TSFM_OK[:2]))]
    for m, gg in s.groupby("model"): a_.plot(gg.bucket, gg.MASE, "o-", label=m, color=MCOL[m])
    a_.set_title(f"{kind} at forecast origin (h={HS[0]})"); a_.set_ylabel("MASE")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{CFG['OUT']}/fig4_missingness.png", dpi=200); plt.close()


## Section XII: Synthesis, Reproducibility Package, & IEEE References

The execution pipeline generates a consolidated report (`summary.txt`) containing all point accuracy metrics, event detection scores, Diebold-Mariano test results, and interval coverage statistics, packaged into `results/` for distribution.

---

### Conclusion & Scientific Insights
1. **Zero-Shot Transfer vs. Local Training:** Pretrained foundation models demonstrate strong competitive zero-shot capability, but domain-trained supervised gradient-boosted trees and pooled multi-horizon LSTMs maintain advantages in capturing local meteorology and persistence dynamics.
2. **Regime Robustness:** Under extreme regime shifts (e.g. 2020 COVID-19 lockdown), models conditioned on recent localized trend features adapt more rapidly than static autoregressive representations.
3. **Conformal Uncertainty Quantification:** Standard Gaussian or uncalibrated native intervals under-cover during winter smog peaks. Stratified and scaled conformal prediction methods successfully restore theoretical coverage guarantees.

---

### References (IEEE Format)
* [1] A. Ansari, L. Stella, C. Turkmen, et al., "Chronos: Learning the Language of Time Series," *Transactions on Machine Learning Research (TMLR)*, 2024.
* [2] Central Pollution Control Board (CPCB), "National Ambient Air Quality Status & Trends in India," Ministry of Environment, Forest and Climate Change, Govt. of India, 2020.
* [3] A. N. Angelopoulos and S. Bates, "A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification," *arXiv preprint arXiv:2107.07511*, 2021.
* [4] F. X. Diebold and R. S. Mariano, "Comparing Predictive Accuracy," *Journal of Business & Economic Statistics*, vol. 13, no. 3, pp. 253–263, 1995.
* [5] D. I. Harvey, S. J. Leybourne, and P. Newbold, "Testing the Equality of Prediction Mean Squared Errors," *International Journal of Forecasting*, vol. 13, no. 2, pp. 281–291, 1997.
* [6] S. Holm, "A Simple Sequentially Rejective Multiple Test Procedure," *Scandinavian Journal of Statistics*, vol. 6, no. 2, pp. 65–70, 1979.
* [7] R. J. Hyndman and A. B. Koehler, "Another Look at Measures of Forecast Accuracy," *International Journal of Forecasting*, vol. 22, no. 4, pp. 679–688, 2006.
* [8] S. A. Stouffer, E. A. Suchman, L. C. DeVinney, S. A. Star, and R. M. Williams Jr, "The American Soldier: Adjustment During Army Life (Studies in Social Psychology in World War II, Vol. 1)," Princeton University Press, 1949.

In [ ]:
# %% [11] summary.txt (paste this into the chat) + zip download
def fmt(df, **kw): return df.to_string(index=False, float_format=lambda x: f"{x:.3f}", **kw)
L = []
L.append(f"RUN: cities={CITIES}\nfolds={[f['name'] for f in FOLDS]} horizons={HS} device={DEVICE} foundation_models={TSFM_OK} mock={CFG['MOCK_TSFM']}")
L.append(f"versions: python {platform.python_version()} numpy {np.__version__} pandas {pd.__version__} torch {getattr(torch,'__version__',None)}")
L.append(f"common test rows per model: {len(E)//len(MODELS)} | city-folds: {sum(VALID.values())}/{len(VALID)}\n")
L.append("=== POINT ACCURACY (MASE macro over cities, 95% CI over cities; skill = 1 - MAE/MAE_persistence; ev_* = detecting days above the city's TRAIN 90th percentile, with its own 95% CI over cities and the pooled positive/total day count n_pos/n_days) ===")
L.append(fmt(PT[["h", "model", "MASE", "MASE_lo", "MASE_hi", "MAE", "RMSE", "skill", "skill_lo", "skill_hi",
                  "ev_recall", "ev_recall_lo", "ev_recall_hi", "ev_prec", "ev_prec_lo", "ev_prec_hi",
                  "ev_f1", "ev_f1_lo", "ev_f1_hi", "n_pos", "n_days"]].sort_values(["h", "MASE"])))
L.append("\n=== EVENT DETECTION BY FOLD (h=1; n_pos/n_days shows how diluted the positive class is per fold, esp. the 2020 lockdown fold) ===")
evf = EV_FOLD[(EV_FOLD.h == HS[0]) & (EV_FOLD.model.isin(["persist", "hgb", "lstm"] + TSFM_OK))]
L.append(fmt(evf[["fold", "model", "ev_recall", "ev_prec", "ev_f1", "n_pos", "n_days"]].sort_values(["fold", "model"])))
L.append("\n=== DIEBOLD-MARIANO vs persistence (Holm-adjusted per city, plus a pooled Stouffer's-method one-sided test across cities: stouffer_p is the pooled probability that the model is NOT better than persistence) ===")
L.append(fmt(DMS[["h", "model", "cities_sig_better", "cities_sig_worse", "n_cities", "stouffer_z", "stouffer_p"]].sort_values(["h", "model"])))
for a in CFG["ALPHAS"]:
    for h in (HS[0], HS[-1]):
        s = CV[(CV.alpha == a) & (CV.h == h)]
        s = s[s.model.isin(["persist", "hgb", "lstm"] + TSFM_OK)]
        L.append(f"\n=== COVERAGE (nominal {int((1-a)*100)}%) h={h}: rows=model/method, cols=stratum; cell = coverage [width], with 95% CI over cities available in coverage.csv ===")
        t = s.assign(cell=s.apply(lambda r: f"{r['cov']:.2f} ({r.cov_lo:.2f} to {r.cov_hi:.2f}) [{r.width:.0f}]", axis=1)).pivot_table(index=["model", "method"], columns="stratum", values="cell", aggfunc="first")
        L.append(t[[c for c in strata if c in t.columns]].to_string())
alpha0 = CFG["ALPHAS"][0]
cvf = CV_FOLD[(CV_FOLD.alpha == alpha0) & (CV_FOLD.h == HS[0]) & (CV_FOLD.stratum == "high_state") & (CV_FOLD.model.isin(["persist", "hgb", "lstm"] + TSFM_OK))]
L.append(f"\n=== HIGH-POLLUTION-DAY COVERAGE BY FOLD (nominal {int((1-alpha0)*100)}%, h=1): shows whether the lockdown fold is masking a coverage failure the same way it masks event detection ===")
L.append(fmt(cvf[["fold", "model", "method", "cov", "cov_lo", "cov_hi", "width", "n"]].sort_values(["fold", "model", "method"])))
L.append("\n=== MISSING DATA at forecast origin: MASE by bucket (h=%d) ===" % HS[0])
L.append(fmt(MS[(MS.h == HS[0])].pivot_table(index=["kind", "bucket"], columns="model", values="MASE").reset_index()))
summary = "\n".join(L); open(f"{CFG['OUT']}/summary.txt", "w").write(summary); print(summary)
with open(f"{CFG['OUT']}/meta.json", "w") as fh: json.dump(dict(cfg={k: (v if not isinstance(v, dict) else v) for k, v in CFG.items()}, cities=CITIES, tsfm=TSFM_OK, minutes=(time.time() - T0) / 60), fh, indent=1, default=str)
import shutil; shutil.make_archive("results_bundle", "zip", CFG["OUT"])
if IN_COLAB:
    from google.colab import files; files.download("results_bundle.zip")
log("DONE. Paste summary.txt above into the chat.")
